In [1]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import config as cnf 

In [2]:
df_train                = pd.read_csv(cnf.path['train_path'])
df_test                 = pd.read_csv(cnf.path['test_path'])
df_gender_submission    = pd.read_csv(cnf.path['gender_submission_path'])

### Логистическая регрессия

In [56]:
df_train_priz = df_train[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')
df_train_priz = pd.get_dummies(df_train_priz, dtype=float, drop_first=True)

X = df_train_priz.drop('Survived', axis=1)
y = df_train_priz['Survived']


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    current_X_train = X.iloc[train_index]
    current_y_train = y.iloc[train_index]
    current_X_test  = X.iloc[test_index]
    current_y_test  = y.iloc[test_index]
    
    scaler  = StandardScaler()
    scaler.fit(current_X_train)
    current_X_train = scaler.transform(current_X_train)
    current_X_test = scaler.transform(current_X_test)
    
    clf = LogisticRegression(random_state=42).fit(current_X_train, current_y_train)
    train_score = clf.score(current_X_train, current_y_train)
    test_score = clf.score(current_X_test, current_y_test)
    
    print("Train_accuracy: ", train_score, "Test_accuracy: ", test_score)

Train_accuracy:  0.8075842696629213 Test_accuracy:  0.770949720670391
Train_accuracy:  0.8036465638148668 Test_accuracy:  0.8033707865168539
Train_accuracy:  0.8064516129032258 Test_accuracy:  0.7865168539325843
Train_accuracy:  0.7980364656381487 Test_accuracy:  0.7808988764044944
Train_accuracy:  0.8022440392706872 Test_accuracy:  0.8258426966292135


In [11]:
def predobr(df_train, df_test, df_gender_submission):
    df_train_priz = df_train[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
    df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
    df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')
    df_train_priz = pd.get_dummies(df_train_priz, dtype=float, drop_first=True)
    
    X = df_train_priz.drop('Survived', axis=1)
    y = df_train_priz['Survived']
    return X, y


def Split(model, X, y, train_index, test_index, scaler=False):
    current_X_train = X.iloc[train_index]
    current_y_train = y.iloc[train_index]
    current_X_test  = X.iloc[test_index]
    current_y_test  = y.iloc[test_index]
    
    if scaler:
        scaler  = StandardScaler()
        scaler.fit(current_X_train)
        current_X_train = scaler.transform(current_X_train)
        current_X_test = scaler.transform(current_X_test)
    
    model.fit(current_X_train, current_y_train)
    train_accuracy = model.score(current_X_train, current_y_train, sample_weight=None)
    test_accuracy = model.score(current_X_test, current_y_test, sample_weight=None)
    
    array_train_accuracy.append(train_accuracy)
    array_test_accuracy.append(test_accuracy)
    
    return array_train_accuracy, array_test_accuracy

def mean_accuracy(array_train_accuracy, array_test_accuracy):
        mean_train_accuracy = float(np.mean(array_train_accuracy))
        mean_train_std = np.std(array_train_accuracy)

        mean_test_accuracy = float(np.mean(array_test_accuracy))
        mean_test_std = np.std(array_test_accuracy)
        print("Train_accuracy: ", mean_train_accuracy, "Test_accuracy: ", mean_test_accuracy)
        
        return mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std


In [12]:
X, y = predobr(df_train, df_test, df_gender_submission)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
array_train_accuracy = []
array_test_accuracy = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):

    model = LogisticRegression(random_state=42)
    array_train_accuracy, array_test_accuracy = Split(model, X, y, train_index, test_index, scaler=True)
    
print("Train_accuracy: ", array_train_accuracy, "Test_accuracy: ", array_test_accuracy)

Train_accuracy:  [0.8075842696629213, 0.8036465638148668, 0.8064516129032258, 0.7980364656381487, 0.8022440392706872] Test_accuracy:  [0.770949720670391, 0.8033707865168539, 0.7865168539325843, 0.7808988764044944, 0.8258426966292135]


In [13]:
X, y = predobr(df_train, df_test, df_gender_submission)

combo_param = list(ParameterGrid(cnf.RandomForestParam))
combo_param_log_reg = list(ParameterGrid(cnf.LogRegParam))
combo_param = combo_param_log_reg + combo_param

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for combo in combo_param:
    name_model = combo['name_model']
    
    if name_model == 'RandomForest':
        n_estimators = combo['n_estimators']
        max_depth = combo['max_depth']
        min_samples_leaf = combo['min_samples_leaf']

        print("Количество деревьев: ", n_estimators, " | Максимальная глубина: ", max_depth, " | Минимум на листе :", min_samples_leaf)

        array_train_accuracy = []
        array_test_accuracy = []

        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, min_samples_leaf=min_samples_leaf)
            array_train_accuracy, array_test_accuracy = Split(model, X, y, train_index, test_index, scaler=False)

        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'n_estimator'         :   n_estimators
            , 'max_depth'           :   max_depth
            , 'min_samples_leaf'    :   min_samples_leaf
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })
    
    elif name_model == 'LogReg':
        
        for i, (train_index, test_index) in enumerate(skf.split(X, y)):

            model = LogisticRegression(random_state=42)
            array_train_accuracy, array_test_accuracy = Split(model, X, y, train_index, test_index, scaler=True)
            
            
        mean_train_accuracy, mean_train_std, mean_test_accuracy, mean_test_std = mean_accuracy(array_train_accuracy, array_test_accuracy)

        results.append({
            'name_model'            :   name_model
            , 'n_estimator'         :   0
            , 'max_depth'           :   0
            , 'min_samples_leaf'    :   0
            , 'train_accuracy'      :   mean_train_accuracy
            , 'train_std'           :   mean_train_std
            , 'test_accuracy'       :   mean_test_accuracy
            , 'test_std'            :   mean_test_std
        })
    

Train_accuracy:  0.80359259025797 Test_accuracy:  0.7935157868307074
Train_accuracy:  0.8035925902579701 Test_accuracy:  0.7935157868307073
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 1
Train_accuracy:  0.8524114754873379 Test_accuracy:  0.828253091456908
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 2
Train_accuracy:  0.8493255275225742 Test_accuracy:  0.8361057058565061
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 3
Train_accuracy:  0.8467986195376398 Test_accuracy:  0.8305002824681438
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 1
Train_accuracy:  0.8709330727894479 Test_accuracy:  0.8316301550436258
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 2
Train_accuracy:  0.8602683707077233 Test_accuracy:  0.828253091456908
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 3
Train_accuracy:  0.8611090974990938 Tes

### Случайный лес

In [ ]:
df_train_priz = df_train[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')
df_train_priz = pd.get_dummies(df_train_priz, dtype=float, drop_first=True)

X = df_train_priz.drop('Survived', axis=1)
y = df_train_priz['Survived']

combo_param = list(ParameterGrid(cnf.param))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for combo in combo_param:
    n_estimators = combo['n_estimators']
    max_depth = combo['max_depth']
    min_samples_leaf = combo['min_samples_leaf']
    
    print("Количество деревьев: ", n_estimators, " | Максимальная глубина: ", max_depth, " | Минимум на листе :", min_samples_leaf)
    
    array_train_accuracy = []
    array_test_accuracy = []
    
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        current_X_train, current_y_train, current_X_test, current_y_test = Split(X, y, train_index, test_index)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, min_samples_leaf=min_samples_leaf)

        model.fit(current_X_train, current_y_train)
        train_accuracy = model.score(current_X_train, current_y_train, sample_weight=None)
        test_accuracy = model.score(current_X_test, current_y_test, sample_weight=None)
        
        array_train_accuracy.append(train_accuracy)
        array_test_accuracy.append(test_accuracy)

    mean_train_accuracy = float(np.mean(array_train_accuracy))
    mean_train_std = np.std(array_train_accuracy)
    
    mean_test_accuracy = float(np.mean(array_test_accuracy))
    mean_test_std = np.std(array_test_accuracy)
    print("Train_accuracy: ", mean_train_accuracy, "Test_accuracy: ", mean_test_accuracy)
    
    results.append({
        'name_model'            :  'RandomForest'
        , 'n_estimator'         :   n_estimators
        , 'max_depth'           :   max_depth
        , 'min_samples_leaf'    :   min_samples_leaf
        , 'train_accuracy'      :   mean_train_accuracy
        , 'train_std'           :   mean_train_std
        , 'test_accuracy'       :   mean_test_accuracy
        , 'test_std'            :   mean_test_std
    })

Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 1
Train_accuracy:  0.8496080022692531 Test_accuracy:  0.8338710689849979
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 2
Train_accuracy:  0.8490458105488756 Test_accuracy:  0.8293704098926622
Количество деревьев:  100  | Максимальная глубина:  5  | Минимум на листе : 3
Train_accuracy:  0.8473612052255858 Test_accuracy:  0.8249011361496453
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 1
Train_accuracy:  0.8720550924247916 Test_accuracy:  0.8349946644906158
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 2
Train_accuracy:  0.8630749956663568 Test_accuracy:  0.8327537505492437
Количество деревьев:  100  | Максимальная глубина:  6  | Минимум на листе : 3
Train_accuracy:  0.8571828167105284 Test_accuracy:  0.8338710689849977
Количество деревьев:  100  | Максимальная глубина:  7  | Минимум на листе : 1
Train_accuracy:  0.893939596

In [14]:
results = pd.DataFrame(results)

In [15]:
results['differenc'] = results['train_accuracy'] - results['test_accuracy']
results.sort_values('test_accuracy', ascending=False).head(50)

,name_model,n_estimator,max_depth,min_samples_leaf,train_accuracy,train_std,test_accuracy,test_std,differenc
14,RandomForest,100,9,1,0.932659,0.004454,0.842854,0.012525,0.089805
19,RandomForest,100,10,3,0.891694,0.003504,0.840613,0.014193,0.051081
17,RandomForest,100,10,1,0.942200,0.001371,0.840606,0.012351,0.101593
16,RandomForest,100,9,3,0.888046,0.004415,0.839483,0.019818,0.048563
18,RandomForest,100,10,2,0.909371,0.002893,0.837248,0.016013,0.072123
12,RandomForest,100,8,2,0.894780,0.004647,0.837236,0.022066,0.057544
11,RandomForest,100,8,1,0.915823,0.006486,0.836125,0.014097,0.079698
3,RandomForest,100,5,2,0.849326,0.003200,0.836106,0.019495,0.013220
8,RandomForest,100,7,1,0.890572,0.004610,0.835001,0.022181,0.055571
13,RandomForest,100,8,3,0.883558,0.005809,0.834995,0.020443,0.048563
